# 🩺 Story Coverage & Recommendation Health — audit

A dedicated, **read-only** audit of Story-Match coverage and the served recommendation feed. It
imports the SAME shared functions the CLI uses
(`examples/audit_story_coverage.py` → `full_report` / `print_report`) — it never reimplements the
analysis, so every number here is byte-for-byte what
`python examples/audit_story_coverage.py --report` prints.

**Two data sources** (cell 2):

- **Golden demo** — a real, connected corpus built by the pipeline itself (the Adams-style
  `story_over_bridge` fixture: catalog + reads + a servable feed). One click, always works — use it
  to see exactly what the report, tables and charts look like.
- **Existing database** — point `DB_URL` at a real store to audit **your** corpus, e.g. a copy of
  your beta's `ih_beta.db`. See **§ Scope & auditing your live beta** at the bottom for how to get
  that file into this notebook.

This notebook makes **no** algorithm changes — it only measures.

In [ ]:
#@title 1 · Setup — clone the repo + install the engine (read-only; no web / server)
import os, sys, subprocess, pathlib

REPO   = "greenwichg/random_walks_with_erasure"  #@param {type:"string"}
BRANCH = "claude/sleepy-gates-oecof1"             #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}         # only if the repo is private

def _in_repo():
    return pathlib.Path("examples/audit_story_coverage.py").is_file()

if not _in_repo():
    auth = (GITHUB_TOKEN + "@") if GITHUB_TOKEN else ""
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://%sgithub.com/%s.git" % (auth, REPO), "app"], check=True)
    os.chdir("app")
print("repo:", pathlib.Path.cwd())

# The auditor builds a REAL recommendation stack (Backend + Personalizer) to diagnose the served
# feed, so it needs the engine library. No web app / server / tunnel — this stays read-only.
try:
    import numpy, scipy, sqlalchemy  # noqa: F401
    print("engine deps already present")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[serve]"], check=True)
print("setup done — run cell 2.")

In [ ]:
#@title 2 · Choose the corpus + build the audit document  (shared functions — no duplicated logic)
#@markdown **Golden demo** audits a real pipeline-built corpus out of the box. **Existing database**
#@markdown audits a real store: set `DB_URL` to a copy of your beta's `ih_beta.db` (see the closing
#@markdown section for how) and `USER_ID` to your reader id (`--list-users` prints it).
SOURCE  = "Golden demo"  #@param ["Golden demo", "Existing database"]
DB_URL  = ""             #@param {type:"string"}
USER_ID = 0              #@param {type:"integer"}
#@markdown `RECS_SOURCE` / `FEED_MAX_PER_OUTLET` apply to **Existing database** only — set them to the
#@markdown same values your engine runs with so the served-feed section mirrors the live app.
RECS_SOURCE         = "feed"  #@param ["feed", "synthetic", "qbias"]
FEED_MAX_PER_OUTLET = 40      #@param {type:"integer"}

import os, sys
if "examples" not in sys.path:
    sys.path.insert(0, "examples")
import audit_story_coverage as asc
import evidence_resolver as er
import store as store_mod

if SOURCE == "Existing database" and DB_URL:
    # Mirror the live engine's serve-time env so serve_and_diagnose reproduces the same feed. The
    # coverage numbers are env-independent; only the served-feed/conversion/ranking section uses it.
    os.environ["RWE_RECS_SOURCE"] = RECS_SOURCE
    if FEED_MAX_PER_OUTLET:
        os.environ["RWE_FEED_MAX_PER_OUTLET"] = str(FEED_MAX_PER_OUTLET)
    st = store_mod.Store(DB_URL)
    uid = USER_ID or 1
else:
    # a REAL golden-scenario store (catalog + reads + servable feed) from the pipeline's own builder
    # — no new seeding logic, the same fixture the Recommendation Validation notebook checks
    from rec_pipeline import extract, pipeline
    case = extract.build(pipeline.load_fixture("story_over_bridge"), keep_env=True)
    st, uid = case.store, (USER_ID or case.reader_uid)

er._INDEX_CACHE.update(key=None, index=None)   # never reuse a prior cell's story index
doc = asc.full_report(st, uid)
print(f"store: {st.url}")
print(f"user:  {uid}   reads: {doc['coverage']['reads']}   "
      f"catalog articles: {doc['coverage']['catalogArticles']}")
print(f"verdict: {doc['verdict']['message']}")

In [ ]:
#@title 3 · The canonical text report  (byte-identical to the CLI `--report`)
asc.print_report(doc)

In [ ]:
#@title 4 · Tables  (pandas views over the SAME document)
import pandas as pd
from IPython.display import display

cov, conv, feed = doc["coverage"], doc["conversion"], doc["feed"]
display(pd.DataFrame([{
    "reads": cov["reads"], "catalog articles": cov["catalogArticles"],
    "story clusters": cov["storyClusters"], "multi-publisher": cov["multiPublisherClusters"],
    "Story Coverage Rate %": doc["coverageRatePercent"],
    "Story Conversion Rate %": (conv or {}).get("ratePercent"),
    "verdict": doc["verdict"]["code"],
}]).T.rename(columns={0: "value"}))

if feed:
    display(pd.DataFrame(
        [{"explanation": asc.LABELS.get(k, k), "cards": v}
         for k, v in sorted(feed["byExplanation"].items())]).set_index("explanation"))
    display(pd.DataFrame([feed["byStrategy"]], index=["cards by engine strategy"]).T)

rows = [{"read": p["title"][:46], "sibling": m["headline"][:46] if m["headline"] else m["url"],
         "publisher": m["publisher"], "story": p["storyId"][:14], "fresh": m["fresh"],
         "outcome": m["outcome"]}
        for p in doc["perRead"] for m in p["siblings"]]
if rows:
    display(pd.DataFrame(rows))

missed = [{"headline": (x.get("headline") or x["sibling"])[:46], "publisher": x["publisher"],
           "reason": x["reason"],
           "best rank": (f"{x['ranks'][0]['strategy']} #{x['ranks'][0]['rank']}"
                         f"/top {x['ranks'][0]['cutoff']}" if x["ranks"] else "-"),
           "gap": (x["gap"] if x["gap"] < 10**9 else None)}
          for x in doc["missed"][:10]]
if missed:
    display(pd.DataFrame(missed, index=range(1, len(missed) + 1)))

In [ ]:
#@title 5 · Charts  (single validated hue; the tables above are the accessible view)
import matplotlib.pyplot as plt

SERIES, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e8e7e3"
ORDER = ["story_match", "bridge", "long_tail", "new_publisher", "topic_continuity",
         "coverage_breadth"]                       # fixed identity order — never value-cycled

def _style(ax, title):
    ax.set_facecolor("white")
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

if doc["feed"]:
    items = [(asc.LABELS[k], doc["feed"]["byExplanation"].get(k, 0))
             for k in ORDER if doc["feed"]["byExplanation"].get(k)]
    if items:
        fig, ax = plt.subplots(figsize=(7, 0.5 * len(items) + 1.2), facecolor="white")
        labels, vals = zip(*items)
        bars = ax.barh(labels, vals, height=0.55, color=SERIES)
        ax.invert_yaxis()
        ax.bar_label(bars, padding=4, color=MUTED, fontsize=9)   # selective direct labels
        _style(ax, f"Served feed by explanation type  (n={doc['feed']['served']})")
        ax.set_xlabel("cards", color=MUTED, fontsize=9)
        plt.tight_layout(); plt.show()

ranked = [x for x in doc["missed"] if x["ranks"]][:10]
if ranked:
    fig, ax = plt.subplots(figsize=(7, 0.5 * len(ranked) + 1.2), facecolor="white")
    names = [f"{(x.get('headline') or x['sibling'])[:34]}… ({x['publisher']})" for x in ranked]
    gaps = [x["gap"] for x in ranked]
    bars = ax.barh(names, gaps, height=0.55, color=SERIES)
    ax.invert_yaxis()
    ax.bar_label(bars, padding=4, color=MUTED, fontsize=9)
    _style(ax, "Missed Story Match opportunities — ranks below the nearest slice cutoff")
    ax.set_xlabel("rank gap to the nearest served slice (smaller = closer to serving)",
                  color=MUTED, fontsize=9)
    plt.tight_layout(); plt.show()

stale = [x for x in doc["missed"] if not x["ranks"]]
if stale:
    print(f"{len(stale)} further missed opportunit(ies) excluded by FRESHNESS "
          "(no rank — never candidates); see table 4.")

## Scope & auditing your live beta

**Read-only.** This notebook only reads the store and builds an in-memory recommender to diagnose
the feed; it never writes to your database and makes no algorithm changes.

**Your beta runs in its own Colab runtime.** Each Colab notebook gets its own VM, so this notebook
cannot reach the live SQLite file *inside* your running beta session directly. Two ways to audit
your real corpus:

1. **Copy the DB across.** In your beta notebook, download `data/ih_beta.db` (📁 Files pane → ⋮ →
   Download), then upload it here (📁 Files pane → Upload — it lands at `/content/ih_beta.db`). In
   cell 2 set **SOURCE = `Existing database`** and **DB_URL = `sqlite:////content/ih_beta.db`**
   (four slashes after `sqlite:` = an absolute path). Find your reader id with
   `!python examples/audit_story_coverage.py --list-users` and put it in **USER_ID**.
2. **Or run the CLI inside the beta runtime.** In your beta notebook, add one cell:
   `!python examples/audit_story_coverage.py --report --user <id>` — it prints the exact same
   report this notebook renders (they share `full_report` / `print_report`).

**What the verdict means:**

- `coverage` — your catalog lacks cross-publisher siblings for the stories you read (a corpus gap,
  not a ranking bug).
- `ranking` — siblings exist and are fresh, but lose during ranking (they rank below the slice
  cutoffs).
- `freshness` — siblings exist in the cluster but fall outside the candidate freshness window, so
  they can never be recommended.
- `none` — Story Match is not significantly limited.

The **Top missed opportunities** table/chart shows, for each unserved fresh sibling, its best
per-strategy rank versus that strategy's slice cutoff — i.e. exactly how large a ranking boost it
would take to surface, measured, with **no** change applied.